In [0]:
from pyspark.sql import functions as F

# Ruta del volumen
ruta = "/Volumes/world_bank/costa_rica/raw_data"

# Archivos JSON
archivos = [
    "gdp_growth.json",
    "inflation.json",
    "unemployment.json",
    "population.json"
]

df_bronze = (
    spark.read
    .format("binaryFile")
    .load(f"{ruta}/*.json")
    .select(
        F.col("path").alias("source_file"),
        F.decode(F.col("content"), "UTF-8").alias("raw_json")
    )
)

display(df_bronze)

In [0]:
# Agregar metadatos de ingesta
df_bronze = (
    df_bronze
    .withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "source_system",
        F.lit("World Bank API")
    )
)

display(df_bronze)

In [0]:
# Guardar los datos en Delta Lake
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "world_bank.costa_rica.bronze_economic_indicators"
    )

print("Tabla Bronze creada correctamente")